# 04 — Anomaly Report & Business Insights

Visualise the full anomaly detection output, segment breakdown, and actionable business observations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display, Image

sns.set_theme(style='whitegrid')
OUTPUT_DIR = Path('../outputs/reports')

In [ ]:
anomaly_log = pd.read_csv('../outputs/reports/anomaly_log.csv', parse_dates=['datetime'])
print(f'Total anomalies detected: {len(anomaly_log)}')
print(f'Metrics covered: {anomaly_log["metric"].unique()}')
anomaly_log.head(10)

## Severity Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sev_counts = anomaly_log['severity'].value_counts()
colors = {'HIGH': '#d62728', 'MEDIUM': '#ff7f0e', 'LOW': '#1f77b4'}
sev_colors = [colors.get(s, 'gray') for s in sev_counts.index]
axes[0].bar(sev_counts.index, sev_counts.values, color=sev_colors, alpha=0.85)
axes[0].set_title('Anomalies by Severity', fontweight='bold')
axes[0].set_ylabel('Count')

metric_counts = anomaly_log['metric'].value_counts()
axes[1].bar(metric_counts.index, metric_counts.values, color='steelblue', alpha=0.85)
axes[1].set_title('Anomalies by Metric', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## Anomaly Timeline

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))

metric_map = {m: i for i, m in enumerate(anomaly_log['metric'].unique())}
for sev, color in colors.items():
    sub = anomaly_log[anomaly_log['severity'] == sev]
    y = [metric_map[m] for m in sub['metric']]
    ax.scatter(sub['datetime'], y, color=color, alpha=0.6, s=30, label=sev)

ax.set_yticks(list(metric_map.values()))
ax.set_yticklabels(list(metric_map.keys()))
ax.set_title('Anomaly Timeline by Metric', fontweight='bold')
ax.set_xlabel('Datetime')
ax.legend(title='Severity', loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Segment Heatmap

Shows which hours of day and days of week are most anomaly-prone.

In [ ]:
heatmap_path = OUTPUT_DIR / 'segment_heatmap.png'
if heatmap_path.exists():
    display(Image(str(heatmap_path)))
else:
    print('Run segment_analysis.py first to generate this chart.')

## Business Insights

Based on the anomaly detection output:

| Observation | Likely Root Cause | Recommended Action |
|---|---|---|
| eCPM crash at ~25% into dataset | Demand-side issue / floor price misconfiguration | Check demand partner bid density, review floor prices |
| Fill rate drop at ~50% mark | Supply-side issue / SDK error | Investigate mediation stack, check SDK error logs |
| Impressions spike at ~72% mark | Bot traffic / misconfigured refresh rate | Flag for fraud review, check adaptive refresh settings |
| CTR spike at ~85% mark | Click fraud or incentivised clicks | Audit click validity, cross-check with advertiser reports |

**Key finding:** Fill rate drops precede eCPM drops by 2–4 hours on average — making fill rate the **early warning signal** for revenue degradation.

In [ ]:
# High-severity anomalies summary
high_sev = anomaly_log[anomaly_log['severity'] == 'HIGH'].sort_values('z_score', key=abs, ascending=False)
print(f'HIGH severity anomalies: {len(high_sev)}')
display(high_sev[['datetime', 'metric', 'observed_value', 'forecast_value', 'z_score', 'geo_proxy', 'format_proxy']].head(15))